# 02a: Condition Enrichment — Fuzzy Match Review

This notebook generates fuzzy match candidates for unmapped conditions and provides
a workflow to review and promote good matches into the condition dictionary.

**Workflow:**
1. Generate fuzzy candidates (written to `ref.condition_candidates`)
2. Review candidates by impact (study count) and confidence (score)
3. Promote approved candidates to the dictionary as `manual` entries
4. Optionally export/import via CSV for offline review

In [ ]:
import pandas as pd
from config.settings import get_duckdb_connection
from src.normalize_conditions import (
    generate_fuzzy_candidates,
    promote_candidates,
    export_candidates_csv,
    import_reviewed_csv,
    create_study_conditions,
    get_coverage_stats,
)

conn = get_duckdb_connection()
print("Connected to DuckDB")

## 1. Generate Fuzzy Candidates

In [ ]:
candidates = generate_fuzzy_candidates(conn)
print(f"Generated {len(candidates):,} fuzzy candidates")

## 2. Summary Statistics

In [ ]:
if not candidates.empty:
    print(f"Total candidates: {len(candidates):,}")
    print(f"\nScore distribution:")
    print(candidates['score'].describe().round(1))
    print(f"\nStudy count distribution:")
    print(candidates['study_count'].describe().round(1))
    print(f"\nTotal studies that would gain a mapping if all approved: "
          f"{candidates['study_count'].sum():,}")
else:
    print("No candidates generated — all unmapped conditions either filtered or below score threshold")

## 3. High-Impact Candidates (study_count >= 10)

These affect the most studies — review carefully.

In [ ]:
if not candidates.empty:
    high_impact = candidates[candidates['study_count'] >= 10].sort_values(
        ['study_count', 'score'], ascending=[False, False]
    )
    print(f"{len(high_impact):,} high-impact candidates (study_count >= 10)")
    high_impact

## 4. High-Confidence Candidates (score >= 90)

These are most likely correct and can be batch-approved quickly.

In [ ]:
if not candidates.empty:
    high_conf = candidates[candidates['score'] >= 90].sort_values(
        ['study_count', 'score'], ascending=[False, False]
    )
    print(f"{len(high_conf):,} high-confidence candidates (score >= 90)")
    high_conf

## 5. Promote Candidates

Select rows to promote. Edit the filter below to approve specific candidates.

In [ ]:
# Example: promote all high-confidence, high-impact candidates
# Uncomment and adjust the filter as needed:

# to_approve = candidates[(candidates['score'] >= 90) & (candidates['study_count'] >= 5)]
# promoted = promote_candidates(conn, to_approve)
# print(f"Promoted {promoted} candidates to dictionary")

## 6. CSV Export/Import (Offline Review)

Export pending candidates to CSV, review in a spreadsheet editor,
then import back with status column set to `approved` or `rejected`.

In [ ]:
# Export
# csv_path = export_candidates_csv(conn)
# print(f"Exported to {csv_path}")

In [ ]:
# Import reviewed CSV
# import_reviewed_csv(conn, "../data/reference/condition_candidates_reviewed.csv")

## 7. Verify Coverage After Promotion

In [ ]:
# Re-run study conditions and check coverage
# create_study_conditions(conn)
# stats = get_coverage_stats(conn)
# print(f"\nCondition coverage: {stats['condition_coverage_pct']}%")
# print(f"TA coverage: {stats['ta_coverage_pct']}%")

In [ ]:
conn.close()